In [ ]:
from anngeno import AnnGeno
import polars as pl

## Create coding variant anngeno

first run `python subset_anngeno.py`

### Create metadata and annotations parquets

In [ ]:
ag = AnnGeno(
    "/home/dnanexus/data_dir/dms_anngeno.ag",
    low_mem = True
)
ag

In [ ]:
an = pl.scan_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/annotations.parquet")

coding_vars = an.filter((pl.col("relative_cds_position_is_nan") == 0)| (pl.col("loftee_hc_is_nan") == 0 )).select("id").collect()["id"].unique().to_list()
coding_vars

In [ ]:
vm = pl.read_parquet("/home/dnanexus/data_dir/dms_anngeno.ag/variant_metadata.parquet")

vm = vm.filter(pl.col('id').is_in(coding_vars)).sort("id")
vm = vm.with_columns(pl.arange(0, pl.len(), eager=False).alias("col"))
vm

In [ ]:
coding_an = an.filter(pl.col('id').is_in(coding_vars)).drop('col').collect()

coding_an = vm[['id', 'col']].join(coding_an, on='id', how='inner')
coding_an

In [ ]:
vm.write_parquet("/home/dnanexus/data_dir/dms_coding.ag/variant_metadata.parquet")
coding_an.write_parquet("/home/dnanexus/data_dir/dms_coding.ag/annotations.parquet")

### Write samples zarr array

In [ ]:
import zarr

old_store = zarr.open('/home/dnanexus/data_dir/dms_anngeno.ag/zarr_store', mode='r')
samps = old_store['samples']
samps

In [ ]:
# Create a new Zarr v3 store
new_path = zarr_dir = "/home/dnanexus/data_dir/samples_zarr"
new_store = zarr.open(
    new_path,
    mode='w',
    zarr_version=3
)

# Copy the array into the new store
new_store.create_dataset(
    name='samples',
    shape=samps.shape,
    dtype=samps.dtype,
    chunks=samps.chunks,
    data=samps[...],   # load data into memory
    overwrite=True
)

### Check new anngeno

In [ ]:
ag_new = AnnGeno(
    "/home/dnanexus/data_dir/dms_coding.ag",
    low_mem = True
)

ag_new

In [ ]:
%%time

ag_new.get_region("ENSG00000122299")

In [ ]:
%%time

ag_new.get_many_regions(["ENSG00000122299","ENSG00000073584","ENSG00000009709"])